In [1]:
import pandas as pd
import numpy as np
import glob
import os
from tqdm import tqdm

In [2]:
!pwd

/storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Celegans


# Databases Having Phenotype biologicalprocess relation of celegans

In [3]:
# Monarch/Monarch_final/Celegans/Cele_PhenotypicFeature_BiologicalProcess.csv

In [4]:
BASE_DIR     = '/storage/Arushi/090526_EvoAge/kg_formation/'
MAPPING_DIR  = BASE_DIR + 'data_collection/databases_for_mapping/'
PROC_DIR     = BASE_DIR + 'processed_data/'

# ── Output path ───────────────────────────────────────────────────────────────
!mkdir Celegans_phenotype_molecularfunction
OUT_PATH = BASE_DIR + 'processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Celegans/Celegans_phenotype_molecularfunction/Celegans_phenotype_molecularfunction.csv'

# ── Required output schema ────────────────────────────────────────────────────
REQUIRED_COLS = [
    'head', 'relation', 'tail',
    'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type',
    'head_id_is', 'tail_id_is',
    'head_detail_name', 'tail_detail_name', 'species'
]

mkdir: cannot create directory ‘Celegans_phenotype_molecularfunction’: File exists


# Monarch

In [5]:
monarch = pd.read_csv(PROC_DIR + 'Monarch/Monarch_final/Celegans/Cele_PhenotypicFeature_MolecularFunction.csv')
monarch.columns = monarch.columns.str.lower()
monarch['kg_type'] = 'Generalised'
monarch['species'] = 'C.elegans'

monarch['kg_source'] = 'Monarch_KG'
#

print(f"monarch: {len(monarch):,} rows")
monarch

monarch: 4 rows


,head,relation,tail,head_type,relation_type,tail_type,relation_source,head_detail_name,tail_detail_name,head_taxon_name,tail_taxon_name,head_id_is,tail_id_is,species,kg_type,kg_source
0,WBPhenotype:0000124,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,infores:upheno,enzyme activity reduced,catalytic activity,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,C.elegans,Generalised,Monarch_KG
1,WBPhenotype:0000125,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,infores:upheno,enzyme activity increased,catalytic activity,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,C.elegans,Generalised,Monarch_KG
2,WBPhenotype:0000727,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,infores:upheno,enzyme activity variant,catalytic activity,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,C.elegans,Generalised,Monarch_KG
3,WBPhenotype:0001695,Phenotype_MolecularFunction,GO:0016757,Phenotype,related_to,MolecularFunction,infores:upheno,glycosyltransferase activity variant,glycosyltransferase activity,Caenorhabditis elegans,NaN,WormBasePhenotype,Quick_GO,C.elegans,Generalised,Monarch_KG


# Consolidate into Unified Schem

In [6]:
# List all source DataFrames to include
source_dfs = [
    monarch
    
]

aligned = []
for df in source_dfs:
    df = df.copy()
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])

final_df = pd.concat(aligned, ignore_index=True)
print(f"Consolidated rows: {len(final_df):,}")
final_df

Consolidated rows: 4


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,WBPhenotype:0000124,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,Monarch_KG,Generalised,WormBasePhenotype,Quick_GO,enzyme activity reduced,catalytic activity,C.elegans
1,WBPhenotype:0000125,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,Monarch_KG,Generalised,WormBasePhenotype,Quick_GO,enzyme activity increased,catalytic activity,C.elegans
2,WBPhenotype:0000727,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,Monarch_KG,Generalised,WormBasePhenotype,Quick_GO,enzyme activity variant,catalytic activity,C.elegans
3,WBPhenotype:0001695,Phenotype_MolecularFunction,GO:0016757,Phenotype,related_to,MolecularFunction,Monarch_KG,Generalised,WormBasePhenotype,Quick_GO,glycosyltransferase activity variant,glycosyltransferase activity,C.elegans


# Sanity Check — Distinct Values

In [7]:
for col in ['relation', 'head_type', 'tail_type', 'relation_type', 'kg_source', 'head_id_is', 'tail_id_is']:
    print(f"{col:20s}: {set(final_df[col])}")

relation            : {'Phenotype_MolecularFunction'}
head_type           : {'Phenotype'}
tail_type           : {'MolecularFunction'}
relation_type       : {'related_to'}
kg_source           : {'Monarch_KG'}
head_id_is          : {'WormBasePhenotype'}
tail_id_is          : {'Quick_GO'}


In [8]:
# Step 4: drop unresolvable rows
before = len(final_df)
final_df = final_df[~final_df['tail_detail_name'].isna()].reset_index(drop=True)
print(f"Dropped {before - len(final_df):,} unresolvable rows → {len(final_df):,} remaining")

Dropped 0 unresolvable rows → 4 remaining


# NaN Audit (pre-dedup)

In [9]:
true_nan   = final_df.isna().sum()
string_nan = final_df.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


# Deduplication

In [10]:
def merge_sources(x):
    """Combine unique, non-null source labels with '::' delimiter."""
    return '::'.join(sorted(set(x.dropna())))

group_cols = ['head', 'relation', 'tail']

final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type':        'first',
    'relation_type':    'first',
    'tail_type':        'first',
    'kg_source':        merge_sources,
    'kg_type':          merge_sources,   # ← changed from 'first'
    'head_id_is':       'first',
    'tail_id_is':       'first',
    'head_detail_name': 'first',
    'tail_detail_name': 'first',
    'species': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
final_df_dedup.head(3)

Before dedup: 4  |  After dedup: 4


,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,species
0,WBPhenotype:0000124,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,Monarch_KG,Generalised,WormBasePhenotype,Quick_GO,enzyme activity reduced,catalytic activity,C.elegans
1,WBPhenotype:0000125,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,Monarch_KG,Generalised,WormBasePhenotype,Quick_GO,enzyme activity increased,catalytic activity,C.elegans
2,WBPhenotype:0000727,Phenotype_MolecularFunction,GO:0003824,Phenotype,related_to,MolecularFunction,Monarch_KG,Generalised,WormBasePhenotype,Quick_GO,enzyme activity variant,catalytic activity,C.elegans


In [11]:
true_nan   = final_df_dedup.isna().sum()
string_nan = final_df_dedup.apply(lambda col: col.astype(str).str.upper().eq('NAN').sum())

pd.DataFrame({
    'NaN_count':          true_nan,
    "'NAN'_string_count": string_nan,
    'Total_NaN_like':     true_nan + string_nan
})

,NaN_count,'NAN'_string_count,Total_NaN_like
head,0,0,0
relation,0,0,0
tail,0,0,0
head_type,0,0,0
relation_type,0,0,0
tail_type,0,0,0
kg_source,0,0,0
kg_type,0,0,0
head_id_is,0,0,0
tail_id_is,0,0,0


In [12]:
print("kg_source values present:", set(final_df_dedup['kg_source']), final_df_dedup['kg_source'].value_counts())

kg_source values present: {'Monarch_KG'} kg_source
Monarch_KG    4
Name: count, dtype: int64


In [13]:
print("kg_source values present:", set(final_df_dedup['kg_type']), final_df_dedup['kg_type'].value_counts())

kg_source values present: {'Generalised'} kg_type
Generalised    4
Name: count, dtype: int64


In [14]:
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 4 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/OTHER_SPECIES/Celegans/Celegans_phenotype_molecularfunction/Celegans_phenotype_molecularfunction.csv


In [15]:
#